# Hybrid RAG: Banking/AML Compliance Assistant

This notebook implements a hybrid retrieval-augmented generation (RAG) pipeline over two banking/AML compliance documents, using Azure OpenAI for embeddings and generation.

**Source documents:**
1. Wolfsberg Group Country Risk FAQ (2024) — structured Q&A
2. Wolfsberg Group Risk-Based Approach (RBA) Guidance (June 2026) — narrative guidance

**Pipeline overview:**

1. Load and chunk both source PDFs, tagging each chunk with document-specific metadata
2. Generate embeddings with local disk caching to avoid redundant API calls
3. Build two retrievers: dense (FAISS, with MMR) and sparse (BM25 keyword search)
4. Fuse both retrievers into a single hybrid retriever (Reciprocal Rank Fusion)
5. Apply a cross-encoder reranker to precisely re-score the final candidate set
6. Support metadata-based filtering for scoped retrieval, including retrieval scoped to a single source document
7. Use a grounding-focused prompt template suited to compliance-style Q&A
8. Cache LLM responses to reduce redundant token usage and latency

Each stage is isolated in its own cell so individual components can be modified, tested, or swapped independently.

## Step 1: Install Dependencies

- `pypdf` — PDF parsing for `PyPDFLoader`
- `rank_bm25` — BM25 sparse/keyword retrieval
- `sentence-transformers` — local cross-encoder inference for reranking
- `rapidfuzz` — fuzzy-matching dependency used internally by some tokenizer/reranker pipelines

In [1]:
!pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken pypdf rank_bm25 sentence-transformers rapidfuzz
!pip install python-dotenv

## Step 2: Imports

In [2]:
import os
import re
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_core.prompts import PromptTemplate
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from sentence_transformers import CrossEncoder

d:\AI Course\Course4 LLM Internal&PlanningSystems\Eval\Banking_RAG_Evals\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3: Load Azure OpenAI Credentials

Credentials are loaded from a `.env` file rather than hardcoded, to keep secrets out of source control.

In [3]:
load_dotenv(override=True)

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

print("Key loaded:", bool(AZURE_OPENAI_API_KEY))

Key loaded: True


## Step 4: Load the Source Documents

Two source PDFs are loaded to satisfy the multi-document retrieval requirement:

1. **Wolfsberg Group Country Risk FAQs (2024)** — structured as numbered Q&A pairs (Q1–Q16)
2. **Wolfsberg Group Risk-Based Approach (RBA) Guidance (June 2026)** — structured as narrative guidance organized under thematic headings (Proportionality, Prioritisation, Effectiveness)

`PyPDFLoader` returns one `Document` per page, with the page number preserved in metadata. Each page is additionally tagged with its originating filename before the documents are combined, so downstream chunking and metadata tagging can distinguish which source document a given chunk came from.

Additional PDFs can be included by adding their file paths to `DOCUMENT_PATHS` below.

In [6]:
DOCUMENT_PATHS = [
    "../data/Wolfsberg Group Country Risk FAQs (2024).pdf",
    "../data/Wolfsberg Group - Risk Based Approach Guidance _June2026.pdf",
]

pages = []
for path in DOCUMENT_PATHS:
    loader = PyPDFLoader(path)
    doc_pages = loader.load()
    for page in doc_pages:
        page.metadata["source_file"] = os.path.basename(path)
    pages.extend(doc_pages)

print(f"Loaded {len(pages)} pages across {len(DOCUMENT_PATHS)} documents")
print(pages[0].metadata)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 66 0 (offset 0)


Loaded 19 pages across 2 documents
{'producer': 'macOS Version 14.2.1 (Build 23C71) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240304080303Z00'00'", 'title': 'Wolfsberg Country Risk FAQs', 'author': 'The Wolfsberg Group', 'moddate': "D:20240304080303Z00'00'", 'source': '../data/Wolfsberg Group Country Risk FAQs (2024).pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'Wolfsberg Group Country Risk FAQs (2024).pdf'}


## Step 5: Chunk the Documents and Tag Metadata

`RecursiveCharacterTextSplitter` is used instead of a single fixed-separator splitter because both source documents contain long, multi-paragraph sections. The splitter attempts natural boundaries first (question markers, then paragraph breaks, then sentence breaks), preserving semantic structure as much as possible while respecting the target chunk size.

`chunk_overlap=200` reduces the risk of a chunk boundary cutting off context needed to interpret dense, cross-referencing regulatory text.

Each chunk is tagged with metadata based on its source document:
- `source_file` — originating PDF filename
- `doc_type` — a normalized label per document (`country_risk_faq` or `rba_guidance`)
- `question_number` — parsed only for the FAQ document, which uses a numbered Q&A structure; left as `None` for the RBA guidance document, which does not follow this structure

This metadata enables scoped retrieval — for example, restricting a query to a single source document, or to a specific question range within the FAQ.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\nQ", "\n\n", "\n", ". ", " "]
)
docs = text_splitter.split_documents(pages)

# Maps source filename to a normalized doc_type label and whether it uses a Q&A structure
DOC_TYPE_MAP = {
    "Wolfsberg_Group_Country_Risk_FAQs__2024_.pdf": {
        "doc_type": "country_risk_faq",
        "has_question_numbers": True,
    },
    "Wolfsberg_Group_-_Risk_Based_Approach_Guidance__June2026.pdf": {
        "doc_type": "rba_guidance",
        "has_question_numbers": False,
    },
}

def tag_metadata(chunks):
    """Tag each chunk with source-specific metadata: doc_type, and question_number where applicable."""
    current_q = None
    for chunk in chunks:
        source_file = chunk.metadata.get("source_file", "")
        doc_info = DOC_TYPE_MAP.get(source_file, {"doc_type": "unknown", "has_question_numbers": False})

        chunk.metadata["doc_type"] = doc_info["doc_type"]

        if doc_info["has_question_numbers"]:
            match = re.search(r"Q(\d{1,2})\.", chunk.page_content)
            if match:
                current_q = int(match.group(1))
            chunk.metadata["question_number"] = current_q
        else:
            chunk.metadata["question_number"] = None

    return chunks

docs = tag_metadata(docs)

print(f"Created {len(docs)} chunks")
for source_file in DOC_TYPE_MAP:
    count = sum(1 for d in docs if d.metadata.get("source_file") == source_file)
    print(f"  {source_file}: {count} chunks")

print("\n--- Example chunk ---")
print("Metadata:", docs[5].metadata)
print(docs[5].page_content[:300])

Created 95 chunks
  Wolfsberg_Group_Country_Risk_FAQs__2024_.pdf: 0 chunks
  Wolfsberg_Group_-_Risk_Based_Approach_Guidance__June2026.pdf: 0 chunks

--- Example chunk ---
Metadata: {'producer': 'macOS Version 14.2.1 (Build 23C71) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240304080303Z00'00'", 'title': 'Wolfsberg Country Risk FAQs', 'author': 'The Wolfsberg Group', 'moddate': "D:20240304080303Z00'00'", 'source': '../data/Wolfsberg Group Country Risk FAQs (2024).pdf', 'total_pages': 13, 'page': 1, 'page_label': '2', 'source_file': 'Wolfsberg Group Country Risk FAQs (2024).pdf', 'doc_type': 'unknown', 'question_number': None}
There is a significant debate as to whether financial crime country risk assessments should be considered a model, a methodology, a tool or even an application (see Question 5). These FAQs use the term methodology for consistency, unless explicit reference is being made to a model. Q1. What is count


## Step 6: Configure Cached Embeddings

`CacheBackedEmbeddings` wraps the Azure embeddings client with a local disk cache keyed by content hash.
Re-running this notebook, or adding new chunks to an existing set, only triggers embedding calls for
new or changed content — previously embedded chunks are served from cache.

In [8]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
    deployment="text-embedding-3-small"
)

store = LocalFileStore("./embedding_cache/")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, store, namespace="text-embedding-3-small"
)

d:\AI Course\Course4 LLM Internal&PlanningSystems\Eval\Banking_RAG_Evals\venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


## Step 7: Build or Load the FAISS Vector Store

The FAISS index is persisted to disk under a name reflecting the combined document set. If the set of source documents changes (e.g., a new PDF is added), the index directory should be deleted, or renamed, so it is rebuilt rather than loaded from a stale cache.

In [9]:
FAISS_INDEX_PATH = "faiss_index_wolfsberg_multi"  # rebuilt automatically if this path does not yet exist

if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS index from disk...")
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH, cached_embeddings, allow_dangerous_deserialization=True
    )
else:
    print("Building new FAISS index...")
    vectorstore = FAISS.from_documents(docs, cached_embeddings)
    vectorstore.save_local(FAISS_INDEX_PATH)

print("Vector store ready.")

Building new FAISS index...
Vector store ready.


## Step 8: Build the BM25 Retriever

BM25 performs keyword-based scoring, weighted by term rarity, and complements dense embedding retrieval:
embeddings capture semantic similarity, while BM25 captures exact term matches (e.g., regulatory acronyms,
proper nouns) that embeddings can sometimes blur together. BM25 runs entirely in-memory with no external
API calls.

In [10]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 5  # how many chunks BM25 returns before fusion

## Step 9: Fuse Dense and Sparse Retrieval

`EnsembleRetriever` runs both retrievers and merges results using Reciprocal Rank Fusion, with results
de-duplicated across methods. The `weights` parameter controls the relative contribution of each method.

The FAISS retriever is configured with `search_type="mmr"` (Maximal Marginal Relevance), which reduces
redundant or near-duplicate results by penalizing candidates too similar to those already selected.

In [11]:
faiss_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 15, "lambda_mult": 0.5}
    # fetch_k: how many candidates MMR considers before picking the final k
    # lambda_mult: 1.0 = pure relevance, 0.0 = pure diversity; 0.5 balances both
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

## Step 10: Metadata-Based Filtering (Optional)

With two source documents now sharing a single vector store, metadata filtering can scope retrieval to a single document (via `doc_type` or `source_file`), or, within the FAQ document specifically, to a range of question numbers. Two example filtered retrievers are shown below.

In [12]:
# Example 1: scope retrieval to a single source document
rba_only_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": {"doc_type": "rba_guidance"}
    }
)

# Example 2: scope retrieval to a specific question range within the FAQ document
faq_question_range_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": lambda meta: meta.get("question_number") in [11, 12]
    }
)

# Not wired into the default pipeline; pass e.g. retriever=rba_only_retriever
# to ask() to use scoped retrieval instead of the full hybrid retriever.

## Step 11: Reranker (Cross-Encoder)

The hybrid retriever returns candidates ranked by an approximate fusion score. A cross-encoder reranker
re-scores each (query, chunk) pair jointly, providing higher precision than similarity comparisons between
independently computed embeddings. Reranking is applied only to the retrieved shortlist, not the full
corpus, due to its higher per-pair computational cost.

`BAAI/bge-reranker-v2-m3` is used here: an Apache 2.0 licensed, multilingual cross-encoder suited to
production use, run locally via `sentence-transformers`'s `CrossEncoder` class with no external API
dependency.

In [13]:
reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, candidates, top_n=3):
    """Re-score retrieved chunks against the query and return the top_n most relevant."""
    pairs = [(query, doc.page_content) for doc in candidates]
    scores = reranker_model.predict(pairs)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored[:top_n]]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4370.21it/s]


## Step 12: Prompt Template

The prompt constrains the model to answer strictly from retrieved context, explicitly instructs it to
decline when the answer is not present, and avoids speculative claims (e.g., naming specific countries)
not stated in the source material — consistent with the compliance domain this document covers.

In [14]:
qa_prompt = PromptTemplate.from_template(
    "You are a compliance assistant answering questions using only the provided "
    "excerpts from AML/CTF guidance documents.\n\n"
    "Rules:\n"
    "- Answer only using the context below. Do not use outside knowledge.\n"
    "- If the answer is not fully contained in the context, say: "
    "\"I don't have enough information in the provided documents to answer that.\"\n"
    "- Do not name specific countries, entities, or make risk determinations "
    "not explicitly stated in the context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

## Step 13: LLM Initialization and Response Caching

Two caching strategies are supported, controlled by the `USE_SEMANTIC_CACHE` flag:

- **`InMemoryCache` (default)** — exact-match caching. If the same fully-formatted prompt is sent
  twice in a session, the second call is served from cache instead of re-invoking the model. Requires
  no external infrastructure, but does not catch paraphrased queries and does not persist across
  sessions.
- **`RedisSemanticCache` (optional)** — paraphrase-aware caching. Incoming queries are embedded and
  compared against previously cached query embeddings; a cache hit is returned if similarity exceeds
  `score_threshold`. This better reflects production RAG caching, but requires a running Redis Stack
  instance with the RediSearch module (plain Redis is not sufficient).

`InMemoryCache` is used by default since it requires no additional setup. `RedisSemanticCache` can be
enabled by setting `USE_SEMANTIC_CACHE = True` once a Redis Stack instance is available.

Note: `gpt-5-mini` is a reasoning-family model and does not support a configurable `temperature`
parameter.

In [15]:
# Step 13: LLM Initialization and Response Caching
USE_SEMANTIC_CACHE = False  # set True if Redis is running locally

if USE_SEMANTIC_CACHE:
    from langchain_community.cache import RedisSemanticCache
    set_llm_cache(RedisSemanticCache(
        redis_url="redis://localhost:6379",
        embedding=embeddings,
        score_threshold=0.95
    ))
else:
    set_llm_cache(InMemoryCache())

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
    deployment_name="gpt-5-mini"
)

## Step 14: Assemble the Pipeline

The retrieval, reranking, and generation stages are composed manually rather than through a prebuilt chain,
since reranking is not natively supported by LangChain's standard `RetrievalQA` chain. This structure also
keeps each stage independently testable.

In [16]:
def ask(query, top_n=3, retriever=None):
    """Run the full hybrid RAG pipeline for a single query."""
    retriever = retriever or hybrid_retriever

    # 1. Hybrid retrieval (BM25 + FAISS/MMR fused)
    candidates = retriever.invoke(query)

    # 2. Rerank candidates with the cross-encoder
    top_docs = rerank(query, candidates, top_n=top_n)

    # 3. Build context string from the final chunks
    context = "\n\n".join(doc.page_content for doc in top_docs)

    # 4. Format the prompt and call the LLM
    formatted_prompt = qa_prompt.format(context=context, question=query)
    response = llm.invoke(formatted_prompt)

    return {
        "answer": response.content,
        "source_documents": top_docs
    }

## Step 15: Query the Pipeline

In [17]:
query = "What are the three key elements of a risk-based approach, and how does country risk factor into prioritisation?"
result = ask(query)

print("Answer:", result["answer"])

Answer: The three key elements are: proportionality, prioritisation and effectiveness.

Country risk feeds into prioritisation because FIs treat country-related factors as contributors to a customer’s financial crime risk. Country risk is described as the additional risk created by investing in, or lending cross‑border to, a foreign country, and FIs identify country risk factors (among others) when assessing customer risk. Examples of country-related factors given include a customer’s domicile, country of incorporation, centre of activity or other nexus such as country of tax residency.


## Step 16: Inspect Retrieved Sources

Source metadata (question number, page number) is printed alongside each retrieved chunk to support auditing of which section of the source document informed a given answer.

In [18]:
print("\n--- Sources used ---")
for i, doc in enumerate(result["source_documents"], 1):
    source_file = doc.metadata.get("source_file", "unknown")
    doc_type = doc.metadata.get("doc_type", "unknown")
    page = doc.metadata.get("page")
    q_num = doc.metadata.get("question_number")

    label = f"Q{q_num}" if q_num is not None else "N/A"

    print(f"Source {i} | {source_file} | doc_type: {doc_type} | question: {label} | page {page}")


--- Sources used ---
Source 1 | Wolfsberg Group - Risk Based Approach Guidance _June2026.pdf | doc_type: unknown | question: N/A | page 0
Source 2 | Wolfsberg Group - Risk Based Approach Guidance _June2026.pdf | doc_type: unknown | question: N/A | page 3
Source 3 | Wolfsberg Group Country Risk FAQs (2024).pdf | doc_type: unknown | question: N/A | page 1
